In [1]:
import os
from typing import List, Dict
import csv
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.core import Feedback
from dotenv import load_dotenv
from trulens.apps.custom import instrument
from openai import OpenAI
import google.generativeai as genai

In [2]:
import vertexai

from vertexai.generative_models import GenerativeModel

In [3]:
vertexai.init(location="us-central1",project="gen-lang-client-0202713877", api_key=os.getenv("GEMINI_API_KEY"))


In [4]:
# with open("../../GroundTruths_Dataset -No Multihop No yes or no questions.csv", mode='r', encoding='utf-8') as file:
#     csv_reader = csv.DictReader(file)
#     # Iterate through rows as dictionaries
#     queries = []
#     for row in csv_reader:
#         queries.append(row["query"]) 
with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 

In [5]:
len(queries)

23

In [6]:
queries[15]

'What are the requirement documents for the good standing certificate of medical staff in the sector the is fee-exempt for renewal staff licenses?'

In [7]:
load_dotenv()


True

In [8]:
from trulens.core import TruSession


session = TruSession()

## Uncomment the following to reset database 
# session.reset_database()

🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [9]:
# pip install pinecone[grpc]
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [27]:
llm = OpenAI(api_key=os.getenv("OPEN_AI_EVAL_KEY"))
gen_llm = GenerativeModel(
  model_name="gemini-1.5-pro-002",
  system_instruction="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points",
)

In [28]:
embed = llm.embeddings.create


AttributeError: 'OpenAI' object has no attribute 'embeddings'

In [29]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=5,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            return docs

In [30]:
class generator:
    def __init__(self, llm):
        self.llm = llm
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate_content(query+formatted_context)
        return response.text

In [31]:
ret = retriever(embed, index)
gen = generator(gen_llm)


In [32]:
res = gen.generate(queries[-1], ret.get_data(queries[-1]))

In [16]:
res

'The provided text outlines requirements and procedures for obtaining advertising licenses from the UAE Ministry of Health and Prevention for healthcare institutions and pharmaceutical groups, but it does not explicitly detail specific content restrictions for medical advertising on websites and social media. Additionally, the document doesn’t offer information on licensing differences between various platforms.  More information is needed to answer your question accurately. \n'

In [33]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response

In [34]:
rag_app = Rag_app(gen, ret)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [36]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="NAIVE RAG",
    app_version="gemini_1.5_pro-large_3-500-Multihop+Yes-NO-4",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [ ]:
# rag_app.query(queries[0])

In [37]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [25]:
queries[20]

'Can pharmaceutical companies export narcotic drugs and what are the validity requirements for such permits?'

In [43]:
with tru_rag as recording:
    for eval in queries[21:]:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

Are there specific requirements for medical professionals over 60 years old and what documentation is needed for their continued practice?


What restrictions apply to medical advertising on websites and social media, and how does the licensing differ between platforms?


In [40]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...
Dashboard already running at path:   Network URL: http://192.168.1.12:9159



<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>